In [1]:
from pathlib import Path

from atlas.common.config.loader import get_settings
from atlas.common.spark.session import get_spark_session

project_root = Path.cwd()

while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
settings = get_settings(
    project_root / "configs" / "base.yaml",
    project_root / "configs" / "local.yaml",
    project_root / "pyproject.toml",
)


In [2]:
spark = get_spark_session(settings.spark, settings.storage, settings.application.name)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/26 19:47:08 WARN Utils: Your hostname, Saileshs-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.6 instead (on interface en0)
26/08/26 19:47:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/saileshpola/Desktop/AtlasProject/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/saileshpola/.ivy2.5.2/cache
The jars for the packages stored in: /Users/saileshpola/.ivy2.5.2/jars
io.delta#delta-spark_2.13 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-12154b81-095e-470b-a95c-ad7f4bbb00d0;1.0
	confs: [default]
	found io.delta#delta-spark_2.13;4.0.0 in central
	found io.del

In [3]:
customer_cdc_stream = (spark.readStream.format("kafka")
                       .option("kafka.bootstrap.servers", settings.kafka.bootstrap_servers)
                       .option("subscribe", "atlas.customer.public.customers")
                       .option("startingOffsets", "earliest")
                       .option("includeHeaders", "true")
                       .load()
                       )

In [4]:
customer_cdc_stream.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)
 |-- headers: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- key: string (nullable = true)
 |    |    |-- value: binary (nullable = true)



In [5]:
customer_converted = customer_cdc_stream.selectExpr("CAST(key as String) AS raw_key"
                                                    , "CAST(value as String) AS raw_value",
                                                    " topic AS kafka_topic", "partition AS kafka_partition",
                                                    "offset AS kafka_offset", "timestamp AS kafka_timestamp",
                                                    "headers AS kafka_headers",)

In [6]:
from pyspark.sql import functions as F

customer_bronze = (customer_converted
             .withColumn("is_tombstone", F.when(F.col("raw_value").isNull(), True).otherwise(False))
             .withColumn("ingested_at", F.current_timestamp() )
             )
customer_bronze.printSchema()

root
 |-- raw_key: string (nullable = true)
 |-- raw_value: string (nullable = true)
 |-- kafka_topic: string (nullable = true)
 |-- kafka_partition: integer (nullable = true)
 |-- kafka_offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- kafka_headers: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- key: string (nullable = true)
 |    |    |-- value: binary (nullable = true)
 |-- is_tombstone: boolean (nullable = false)
 |-- ingested_at: timestamp (nullable = false)



In [7]:
from atlas.common.paths.loader import get_paths

paths = get_paths(settings)
bronze_customer_path = paths.bronze_path("customer/cdc/customers/notebook")
bronze_customer_checkpoint = paths.checkpoint_path("customer/cdc/customers/notebook")
print(bronze_customer_checkpoint)
print(bronze_customer_path)

/Users/saileshpola/Desktop/AtlasProject/data/checkpoint/customer/cdc/customers/notebook
/Users/saileshpola/Desktop/AtlasProject/data/lakehouse/bronze/customer/cdc/customers/notebook


In [8]:
customer_bronze_to_write = customer_bronze.withColumn("ingested_date", F.to_date("ingested_at"))
query = (
    customer_bronze_to_write.writeStream.format("parquet")
    .outputMode("append")
    .option("checkpointLocation", bronze_customer_checkpoint)
    .partitionBy("ingested_date")
    .trigger(availableNow=True)
    .start(bronze_customer_path)
)

26/08/26 19:47:14 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
